In [2]:
from google.colab import files

uploaded = files.upload()

Saving fruit_dataset.zip to fruit_dataset.zip


In [4]:
from zipfile import ZipFile

with ZipFile("fruit_dataset.zip", "r") as zip_ref:
    zip_ref.extractall()

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [5]:
import os

for root, dirs, files in os.walk("fruit_dataset"):
    print(root)

fruit_dataset
fruit_dataset/fresh fruits
fruit_dataset/rotten fruits


In [7]:
import os

for root, dirs, files in os.walk("."):
    print(root)

.
./.config
./.config/logs
./.config/logs/2026.06.04
./.config/configurations
./fruit_dataset
./fruit_dataset/fresh fruits
./fruit_dataset/rotten fruits
./sample_data


In [8]:
!find . -type d

.
./.config
./.config/logs
./.config/logs/2026.06.04
./.config/configurations
./fruit_dataset
./fruit_dataset/fresh fruits
./fruit_dataset/rotten fruits
./sample_data


In [9]:
import os

print("Fresh Images:")
print(os.listdir("fruit_dataset/fresh fruits"))

print("\nRotten Images:")
print(os.listdir("fruit_dataset/rotten fruits"))

Fresh Images:
['Banana.jpg', 'Apple.jpg', 'Orange.jpg']

Rotten Images:
['Rotten Apple.jpg', 'Rotten banana.jpg', 'Rotten Orange.jpg']


In [10]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(
    root="fruit_dataset",
    transform=transform
)

In [11]:
print(dataset.classes)

['fresh fruits', 'rotten fruits']


In [12]:
print("Total Images:", len(dataset))

Total Images: 6


In [13]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

print("Train Images:", len(train_dataset))
print("Test Images:", len(test_dataset))

Train Images: 4
Test Images: 2


In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

In [15]:
from torchvision import models

model = models.resnet18(weights="DEFAULT")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 206MB/s]


In [16]:
for param in model.parameters():
    param.requires_grad = False

In [17]:
import torch.nn as nn

model.fc = nn.Linear(512, 2)

In [18]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print(device)

cuda


In [19]:
print(device)

cuda


In [20]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [21]:
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, Loss: {running_loss:.4f}"
    )

Epoch 1/5, Loss: 2.0984
Epoch 2/5, Loss: 1.4603
Epoch 3/5, Loss: 1.5741
Epoch 4/5, Loss: 0.9074
Epoch 5/5, Loss: 1.5076


In [22]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 100.00%


In [23]:
torch.save(
    model.state_dict(),
    "fruit_classifier.pth"
)

print("Model Saved!")

Model Saved!


In [24]:
from google.colab import files

files.download("fruit_classifier.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import os
print(os.listdir())

['.config', 'fruit_dataset', 'fruit_dataset.zip', 'fruit_classifier.pth', 'sample_data']
